# Geolocation Inference from Call Detail Records Using K-Means Clustering

**Course**: OS4601 Advanced Data Analysis — Naval Postgraduate School  
**Author**: Sean Forester  
**Original**: 2018 | **Updated**: 2026

## Objective

Infer the likely **residence** and **workplace** locations of 10 mobile phone users from 3 years of Call Detail Records (CDRs) using temporal filtering and K-Means clustering on cell tower coordinates.

## Approach

1. **Temporal segmentation** — separate weekday working hours from weekend/twilight patterns
2. **Domain heuristics** — twilight weekend calls (0000–0600) cluster near residence; weekday calls (0730–1700) cluster near workplace
3. **K-Means clustering** — identify spatial concentrations in tower coordinates, validated by mean call time per cluster


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_context("notebook", font_scale=1.1)
sns.set_style("ticks")


## Data

A Call Detail Record (CDR) documents the details of a telephone call or other telecommunications transaction — time, duration, completion status, source and destination numbers.

This dataset contains call records for 10 people tracked over 3 years. Each record includes the latitude and longitude of the cell tower that handled the call.


In [ ]:
df = pd.read_csv("data/CDR.csv", na_values="")

df["CallDate"] = pd.to_datetime(df["CallDate"])
df["CallTime"] = pd.to_timedelta(df["CallTime"])
df["Duration"] = pd.to_timedelta(df["Duration"])

print(f"Records: {len(df):,}")
print(f"Date range: {df['CallDate'].min().date()} to {df['CallDate'].max().date()}")
print(f"Unique callers: {df['In'].nunique()}")
df.head()


## All Tower Locations by Caller


In [ ]:
distinct_callers = df["In"].unique()

sns.lmplot(
    x="TowerLon",
    y="TowerLat",
    data=df,
    fit_reg=False,
    hue="In",
    scatter_kws={"marker": "D", "s": 100},
    height=10,
    aspect=1,
)
plt.title("Cell Tower Locations of 10 Mobile Phones")
plt.xlabel("Longitude")
plt.ylabel("Latitude");


## Temporal Segmentation

People behave differently on weekends vs weekdays:

**Weekends** — no commute, sleep in, run errands. Calls between midnight and 0600 almost certainly originate near home.

**Weekdays** — at work 0730–1700, home during early morning and late night, brief commute windows in between.


### Residence: Weekend Twilight Hours (0000–0600)


In [ ]:
user1 = df[df["In"] == distinct_callers[0]]

weekend_calls = user1[user1["DOW"].isin(["Sat", "Sun"])]
twilight_weekends = weekend_calls[
    (weekend_calls["CallTime"] < "06:00:00") | (weekend_calls["CallTime"] > "24:00:00")
]

twilight_size = 100 * (twilight_weekends["Duration"] / twilight_weekends["Duration"].mean())

sns.lmplot(
    x="TowerLon",
    y="TowerLat",
    data=twilight_weekends,
    fit_reg=False,
    hue="In",
    scatter_kws={"marker": "D", "s": twilight_size},
    height=10,
    aspect=1,
)
plt.title("User 1 — Twilight Weekend Calls (0000–0600)")
plt.xlabel("Longitude")
plt.ylabel("Latitude");


In [ ]:
print("Likely residence tower coordinates:")
print(twilight_weekends["TowerLat"].value_counts().head())
print(twilight_weekends["TowerLon"].value_counts().head())


### Workplace: Weekday Working Hours (0730–1700)


In [ ]:
weekday_calls = user1[~user1["DOW"].isin(["Sat", "Sun"])]
weekday_five = weekday_calls[
    (weekday_calls["CallTime"] > "07:30:00") & (weekday_calls["CallTime"] < "17:00:00")
]

sns.lmplot(
    x="TowerLon",
    y="TowerLat",
    data=weekday_five,
    fit_reg=False,
    hue="In",
    scatter_kws={"marker": "D", "s": 80},
    height=10,
    aspect=1,
)
plt.title("User 1 — Weekday Working Hours (0730–1700)")
plt.xlabel("Longitude")
plt.ylabel("Latitude");


## K-Means Clustering

With temporal filtering narrowing the data, K-Means identifies spatial concentrations. For weekday working hours with K=2:
- The **larger cluster** corresponds to the workplace
- The **smaller cluster** captures commute/transit towers

We validate by comparing the **mean call time** of each cluster — the transit cluster should show earlier mean times (commute window).


In [ ]:
def cluster_info(model):
    """Print inertia and per-cluster sample counts."""
    print(f"Inertia: {model.inertia_:.4f}")
    print("-" * 40)
    for i in range(len(model.cluster_centers_)):
        print(f"  Cluster {i}: centroid {model.cluster_centers_[i]}, samples: {(model.labels_ == i).sum()}")


def fit_kmeans(data, k):
    """Fit KMeans on TowerLat/TowerLon columns."""
    coords = data[["TowerLat", "TowerLon"]]
    model = KMeans(n_clusters=k, init="random", n_init=1, algorithm="lloyd", random_state=42)
    model.fit(coords)
    return model


def identify_clusters(model):
    """Return (least_samples_idx, most_samples_idx)."""
    counts = np.array([(model.labels_ == i).sum() for i in range(len(model.cluster_centers_))])
    return counts.argmin(), counts.argmax()


In [ ]:
model = fit_kmeans(weekday_five, k=2)
cluster_info(model)

least_idx, most_idx = identify_clusters(model)
print(f"\nTransit cluster: {least_idx}  |  Workplace cluster: {most_idx}")

for idx, label in [(least_idx, "Transit"), (most_idx, "Workplace")]:
    mask = model.labels_ == idx
    samples = weekday_five.iloc[mask]
    print(f"{label} — mean call time: {samples['CallTime'].mean()}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(weekday_five["TowerLon"], weekday_five["TowerLat"], c="steelblue", marker="o", alpha=0.2)

markers = [("r", "Workplace"), ("gold", "Transit")]
for i, (color, label) in enumerate(markers):
    idx = most_idx if label == "Workplace" else least_idx
    ax.scatter(
        model.cluster_centers_[idx, 1],
        model.cluster_centers_[idx, 0],
        s=200, c=color, marker="x", linewidths=3, label=label,
    )

ax.set_title("K-Means: Weekday Working Hours — User 1")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend();


## All 10 Users


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for i, caller in enumerate(distinct_callers):
    user = df[df["In"] == caller]

    weekday = user[~user["DOW"].isin(["Sat", "Sun"])]
    working = weekday[(weekday["CallTime"] > "07:30:00") & (weekday["CallTime"] < "17:00:00")]

    model = fit_kmeans(working, k=2)
    least_idx, most_idx = identify_clusters(model)

    ax = axes[i]
    ax.scatter(working["TowerLon"], working["TowerLat"], c="steelblue", marker="o", alpha=0.2, s=20)
    ax.scatter(model.cluster_centers_[most_idx, 1], model.cluster_centers_[most_idx, 0],
               s=150, c="r", marker="x", linewidths=2, label="Work")
    ax.scatter(model.cluster_centers_[least_idx, 1], model.cluster_centers_[least_idx, 0],
               s=150, c="gold", marker="x", linewidths=2, label="Transit")
    ax.set_title(f"User {i + 1}", fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("Inferred Workplace Locations — All 10 Users", fontsize=14)
plt.tight_layout();


## Exploratory: Can K-Means Recover Caller Identity?

With K=10 (one cluster per user), can spatial clustering alone separate the 10 phone numbers? This is a sanity check — if users live and work in distinct areas, their tower usage patterns should be spatially separable.


In [ ]:
kmeans_10 = KMeans(n_clusters=10, random_state=42)
kmeans_10.fit(df[["TowerLat", "TowerLon"]])
phone_labels = kmeans_10.predict(df[["TowerLat", "TowerLon"]])

plt.figure(figsize=(10, 10))
plt.scatter(df["TowerLon"], df["TowerLat"], c=phone_labels, s=50, cmap="viridis", alpha=0.5)
plt.colorbar(label="Cluster")
plt.title("K=10 Clustering — Can Tower Usage Recover Caller Identity?")
plt.xlabel("Longitude")
plt.ylabel("Latitude");


## Conclusions

1. **Temporal segmentation is essential** — raw CDR coordinates are noisy; domain-driven time windows isolate meaningful spatial patterns
2. **K-Means with K=2** reliably separates workplace from transit towers, validated by mean call time (transit cluster shows earlier times consistent with morning commute)
3. **Twilight weekend filtering** successfully identifies residence tower clusters
4. **Limitations** — cell tower coordinates are proxies for user location (resolution depends on tower density); K=2 assumes a simple home/work pattern and may miss users with multiple regular destinations
